# 04 - PHASE 1 GATE (Pl@ntNet): reliability and predictability of delta_y

AGENTS.md Sec 6 - the experiment that decides the project. **Nothing under `pcc/method/` may be
built until this passes.**

- **Gate A (6.2)** split-half reliability of delta_y >= 0.30. The CEILING on any achievable R2.
- **Gate B (6.3)** held-out-CLASS R2 clearly > 0, normalized by the ceilings.
- **Gate C (6.3)** geometry must beat log-prevalence-only and distance-only.
- **Sec 6.4** the predicted correction must buy efficiency on held-out classes.

## Amendment 5 governs how this is read - please read before the numbers

Descriptor stability on Pl@ntNet never reached 0.90 (best 0.815 at q=50) and the tail quartile is
**flat at ~0.68** because those classes hold only 2-7 images - no quota can fix it. **Sec 3.3 is
therefore NOT satisfied**, and three consequences follow, all human-approved:

1. **Gate B is read asymmetrically.** Descriptor noise only ATTENUATES R2; it cannot manufacture
   predictability. A **PASS is meaningful**; a **FAIL is AMBIGUOUS**, not evidence of no signal.
2. **Gate C is computed WITHIN prevalence strata (PRIMARY)**, pooled only as a secondary view.
   Descriptor accuracy tracks prevalence (head-tail spread **+0.238**), so the pooled comparison is
   confounded. Validated on planted stories: prevalence-driven delta -> geometry beats prevalence in
   **0/4** strata; geometry-driven delta -> **4/4**. Pooled R2 was 0.935 vs 0.928, i.e. useless for
   telling them apart.
3. **PRIMARY feature set = the 4 features with stability >= 0.90**; `full` is the sensitivity view.

## Amendment 2: delta_y at matched n_cal, plus a prevalence null

The quantile estimator's bias depends on group size and `n_y` tracks prevalence, so unmatched
delta_y correlates with prevalence even with zero class structure. `n_cal=25` is PRIMARY (152
classes), `n_cal=10` the sensitivity (283 classes); both are pre-registered and both reported.

## Sec 6.4: Amendment 4 design, plus a tail-facing view

Efficiency is measured in the **held-out label space** with the threshold vector free to deflate, so
a constant delta_hat is a genuine no-op. Separately, `pcc.eval.tail` reports macro-coverage per
prevalence stratum - the only way to say anything about the tail, where delta_y is unmeasurable but
coverage aggregated over hundreds of classes is not.

**Scores are OURS** (checkpoint gate did not pass - reports/phase0_checkpoint_gate.md).


## 1. Config - `# === EDIT ME ===`


In [ ]:
# === EDIT ME ===========================================================
REPO_URL   = ''
REPO_DIR   = 'foundation-cp'
DRIVE_ROOT = '/content/drive/MyDrive/pcc'

DATASET    = 'plantnet'
BACKBONE   = 'resnet50_ltc'
CAL_SPLIT  = 'cal'            # delta_y / calibration
EVAL_SPLIT = 'test'           # evaluation
DESC_SPLIT = 'train_quota'    # descriptors: TRAINING data only (Sec 6.3)

ALPHA      = 0.1              # alpha=0.05 needs n>=19/class (187 classes), 0.01 needs 99 (57)
N_CAL_PRIMARY     = 25        # Amendment 2, pre-registered pair
N_CAL_SENSITIVITY = 10
N_SPLITS_A  = 100             # split-half repetitions (gate A)
N_SPLITS_BC = 100             # class-level splits (gate B/C)
N_NULL      = 30              # prevalence-null repetitions
N_STRATA    = 4
N_STRATA_POWER = 2            # 76 classes/stratum instead of 38: same confound
                              # controlled, twice the power. Reported ALONGSIDE the
                              # 4-stratum primary, never as a replacement for it.
RUN_RETIRED_64 = True         # keep design 4 as a secondary, non-verdict view
STABLE_THRESHOLD = 0.90       # descriptor-stability cut for the PRIMARY feature set
SEED = 42
EMB_ROOT = f'{DRIVE_ROOT}/embeddings/{DATASET}/{BACKBONE}'
# =======================================================================
print('alpha', ALPHA, '| n_cal primary/sensitivity', N_CAL_PRIMARY, N_CAL_SENSITIVITY)


## 2. Mount Drive + repo + env


In [ ]:
import os, subprocess
from google.colab import drive
drive.mount('/content/drive')
if REPO_URL and not os.path.isdir(REPO_DIR):
    subprocess.run(['git','clone',REPO_URL,REPO_DIR], check=True)
if os.path.isdir(REPO_DIR):
    os.chdir(REPO_DIR if os.path.isabs(REPO_DIR) else '/content/'+REPO_DIR)
os.environ['PYTHONPATH'] = os.getcwd() + os.pathsep + os.environ.get('PYTHONPATH','')
subprocess.run(['pip','install','-q','-r','requirements.txt'], check=False)
from pcc.utils.seed import set_seed; from pcc.utils.io import environment_stamp
set_seed(SEED)
print('env:', environment_stamp()['packages'])


## 3. Load: cal scores, eval scores, train embeddings. LEAK GUARD asserted.


In [ ]:
import numpy as np
from pcc.data.load import load_split, load_scores, split_provenance, per_class_counts
from pcc.data.ltc_datasets import NUM_CLASSES
from pcc.scores.base import thr_lac

K = NUM_CLASSES[DATASET]
prov = split_provenance(EMB_ROOT, CAL_SPLIT)
print('scores_source:', prov.get('scores_source'),
      '| under_gate_exception:', prov.get('under_gate_exception'))

sm_cal, y_cal = load_scores(EMB_ROOT, CAL_SPLIT)
sm_ev,  y_ev  = load_scores(EMB_ROOT, EVAL_SPLIT)
S_cal, S_ev = thr_lac(sm_cal), thr_lac(sm_ev)
s_true_cal = S_cal[np.arange(len(y_cal)), y_cal]

dtr = load_split(EMB_ROOT, DESC_SPLIT)
tr_emb = np.asarray(dtr['embeddings']); tr_lab = np.asarray(dtr['labels']).astype(int)
tr_lg  = np.asarray(dtr['logits'])

cal_counts = per_class_counts(y_cal, K)
tr_counts  = per_class_counts(tr_lab, K)
print(f'cal {len(y_cal)} | eval {len(y_ev)} | descriptor images {len(tr_lab)}')
print(f'cal samples/class  median {int(np.median(cal_counts[cal_counts>0]))}')
print(f'train images/class median {int(np.median(tr_counts[tr_counts>0]))}')

# LEAK GUARD (Sec 8.3): descriptors must come from TRAIN, never the calibration split.
# cal/eval come from the val/test directories; descriptors from the train directory.
assert DESC_SPLIT == 'train_quota', 'descriptors must come from the train split'
assert CAL_SPLIT != DESC_SPLIT and EVAL_SPLIT != DESC_SPLIT
print('leak guard OK: descriptors from TRAIN; cal/eval from val/test (disjoint sources)')


## 4. GATE A - split-half reliability of delta_y. Sets the R2 ceiling.


In [ ]:
from pcc.targets.delta import split_half_reliability
from pcc.eval.stats import mean_ci

relA = split_half_reliability(s_true_cal, y_cal, K, ALPHA,
                              n_splits=N_SPLITS_A, seed=SEED)
sp = relA['reliability_splits']; sp = sp[~np.isnan(sp)]
rel_ci = mean_ci(sp)
r_delta = float(relA['reliability_mean'])
print(f"gate A reliability = {r_delta:.3f}  95% CI "
      f"[{rel_ci['ci_low']:.3f}, {rel_ci['ci_high']:.3f}]   (threshold 0.30)")
gate_A_pass = bool(np.isfinite(rel_ci['ci_low']) and rel_ci['ci_low'] >= 0.30)
print(f"  eligible classes (>=4 cal samples): {relA['n_classes_eligible']}")
print(f"  splits producing a value: {relA['n_splits_with_a_value']}/{N_SPLITS_A}")
print(f"  classes contributing per split (mean): {relA['n_classes_contributing_mean']:.0f}")
if relA.get('undefined_reason'):
    print('  UNDEFINED:', relA['undefined_reason'])
print('gate A:', 'PASS' if gate_A_pass else ('UNDEFINED' if not np.isfinite(r_delta) else 'FAIL'))
print(f'(cal median is {int(np.median(cal_counts[cal_counts>0]))} samples/class, so gate A is'
      f' noise-limited; the level-matched estimator is used so small classes still contribute)')


## 5. delta_y at matched n_cal (Amendment 2) + prevalence null


In [ ]:
from pcc.targets.delta import delta_y, delta_y_matched_n, prevalence_null

deltas, nulls = {}, {}
for tag, ncal in (('primary', N_CAL_PRIMARY), ('sensitivity', N_CAL_SENSITIVITY)):
    dm, kept = delta_y_matched_n(s_true_cal, y_cal, K, ALPHA, n_cal=ncal, seed=SEED)
    deltas[tag] = (np.where(kept, dm, np.nan), kept, ncal)
    nl = prevalence_null(s_true_cal, y_cal, K, ALPHA, n_cal=ncal,
                         n_reps=N_NULL, seed=SEED)
    nulls[tag] = nl
    ok = int((kept & np.isfinite(dm)).sum())
    corr = np.nan
    m = kept & np.isfinite(dm) & (cal_counts > 0)
    if m.sum() > 3:
        corr = float(np.corrcoef(dm[m], np.log(cal_counts[m]))[0,1])
    print(f'[{tag}] n_cal={ncal}: delta_y defined for {ok}/{K} classes | '
          f'corr(delta, log n_y)={corr:+.3f}')
    if nl['n_reps'] == 0:
        print('   prevalence null UNDEFINED:', nl.get('undefined_reason'))
    else:
        print(f"   null: mean {nl['null_mean']:+.3f} sd {nl['null_sd']:.3f} "
              f"|null|p95 {nl['null_abs_p95']:.3f} -> observed corr is "
              f"{'WITHIN' if abs(corr) <= nl['null_abs_p95'] else 'BEYOND'} the null")
d_unm = delta_y(s_true_cal, y_cal, K, ALPHA)
print(f'\nunmatched (all samples) delta_y defined for {int(np.isfinite(d_unm).sum())}/{K}'
      f' - reported as sensitivity only (Amendment 2)')


## 6. Descriptors from TRAIN + the pre-registered feature sets


In [ ]:
from pcc.descriptors.phi import build_descriptors
from pcc.descriptors.stability import descriptor_stability, QUOTA_DETERMINED

Phi, names = build_descriptors(tr_emb, tr_lg, tr_lab, K,
                               log_prevalence_from=tr_counts)
print('Phi', Phi.shape, '| finite rows:', int(np.isfinite(Phi).all(axis=1).sum()))

stab = descriptor_stability(tr_emb, tr_lg, tr_lab, K, quotas=(50,), n_reps=3,
                            seed=SEED, stable_threshold=STABLE_THRESHOLD,
                            method='bootstrap')
pf = stab['by_quota'][50]['per_feature']
r_phi = float(stab['by_quota'][50]['mean_corr'])
stable_names = [nm for nm in names if nm not in QUOTA_DETERMINED
                and np.isfinite(pf.get(nm, np.nan)) and pf[nm] >= STABLE_THRESHOLD]
FEATURE_SETS = {'stable': stable_names, 'full': list(names)}
assert stable_names, 'no descriptor passed the stability screen'
print(f'r_phi (descriptor ceiling, bootstrap q=50) = {r_phi:.3f}')
print('PRIMARY stable set:', stable_names)
print('dropped as unstable:', [nm for nm in names
      if nm not in stable_names and nm not in QUOTA_DETERMINED])
print(f'\nTWO CEILINGS: target r_delta={r_delta:.3f}, descriptor r_phi={r_phi:.3f},'
      f' joint ~{r_delta*r_phi:.3f}. A perfect model cannot exceed the joint bound.')


## 7. GATE B/C - PRIMARY is stratified by prevalence (Amendment 5)


In [ ]:
from pcc.eval.predictability import predictability, predictability_by_stratum

gate_bc = {}
for dtag in ('primary', 'sensitivity'):
    dvec = deltas[dtag][0]
    for fset, feats in FEATURE_SETS.items():
        key = f'{dtag}|{fset}'
        # Amendment 7: pass the COMPLETE Phi and restrict the full model via
        # feature_subset. Slicing Phi to the stable set removed log_prevalence and the
        # distance column, so gate C silently returned None for the PRIMARY set.
        pooled = predictability(Phi, dvec, list(names), feature_subset=feats,
                                reliability=r_delta,
                                n_splits=N_SPLITS_BC, seed=SEED)
        strat = predictability_by_stratum(Phi, dvec, list(names), cal_counts,
                                          feature_subset=feats,
                                          reliability=r_delta, n_splits=N_SPLITS_BC,
                                          seed=SEED, n_strata=N_STRATA)
        # Amendment 5 conditions on prevalence to break the descriptor-quality
        # confound; 2 strata still condition on it, with twice the classes per cell.
        # Justification is POWER (CI widths, derivable without seeing the outcome),
        # not the result -- so the 4-stratum run stays PRIMARY and is reported even
        # when it fails. Choosing 2 after seeing 4 fail would be selection.
        strat2 = predictability_by_stratum(Phi, dvec, list(names), cal_counts,
                                           feature_subset=feats,
                                           reliability=r_delta, n_splits=N_SPLITS_BC,
                                           seed=SEED, n_strata=N_STRATA_POWER)
        gate_bc[key] = {'pooled': pooled, 'stratified': strat,
                        'stratified_power2': strat2}
        r2 = pooled['r2_by_predictor']['full']
        print(f'=== {key} ===')
        print(f"  POOLED (secondary): R2={r2['mean']:+.3f} "
              f"CI [{r2['ci_low']:+.3f},{r2['ci_high']:+.3f}] gate_B={pooled['gate_B_pass']}")
        print(f"  STRATIFIED (PRIMARY): {strat['n_strata_gate_B_pass']}"
              f"/{strat['n_strata_reported']} strata pass gate B; "
              f"geometry beats: {strat['n_strata_full_beats_ablation']}")
        if pooled['underpowered']:
            print(f"  UNDERPOWERED: {pooled['n_features_full']} features on "
                  f"{pooled['n_train_classes']} training classes -> a gate-C FAIL here is"
                  f" not evidence; compare capacity-matched sets instead (Amendment 7)")
        print(f"  gate C ablations run: {sorted(pooled['gate_C_detail'])} "
              f"(distance baseline = {pooled['distance_col_used']})")
        if pooled['distance_baseline_is_nested_in_full']:
            print(f"  NOTE: the distance baseline '{pooled['distance_col_used']}' is ALSO a"
                  f" feature of the full model, so gate C here asks whether the REMAINING"
                  f" features add anything -- strictly harder than pre-registered Sec 6.5C.")
            print(f"        gate_C (nested, harder)   = {pooled['gate_C_pass']}")
            print(f"        gate_C (pre-registered)  = {pooled['gate_C_pass_prereg']}"
                  f"  [independent baseline = {pooled['distance_col_prereg']}]")
        for sname, sv in strat['summary'].items():
            up = ' UNDERPOWERED' if sv['underpowered'] else ''
            w = sv['r2_full_ci'][1] - sv['r2_full_ci'][0]
            print(f"    {sname:18s} n={sv['n_classes']:4d} R2={sv['r2_full']:+.3f} "
                  f"ciW={w:.2f} B={sv['gate_B_pass']} beats={sv['beats']}{up}")
        print(f"  POWER VIEW ({N_STRATA_POWER} strata, reported alongside, not a replacement): "
              f"{strat2['n_strata_gate_B_pass']}/{strat2['n_strata_reported']} pass gate B; "
              f"beats: {strat2['n_strata_full_beats_ablation']}")
        for sname, sv in strat2['summary'].items():
            w = sv['r2_full_ci'][1] - sv['r2_full_ci'][0]
            pg = sv.get('gate_C_pass_prereg')
            print(f"    {sname:18s} n={sv['n_classes']:4d} R2={sv['r2_full']:+.3f} "
                  f"ciW={w:.2f} B={sv['gate_B_pass']} C={sv.get('gate_C_pass')} "
                  f"C_prereg={pg}")


## 8. Sec 6.4 - coverage EQUITY on held-out classes at matched set size (Amendment 8)

Design 4 (match coverage, read set size) is **retired**. Its own recorded controls show why:
`oracle +0.045` against `shuffled oracle -15.39` - a PERFECT delta_hat bought essentially
nothing, so the ceiling was ~0 and the whole dynamic range was negative. Same root cause as
the Phase-0 criterion: set size is near-vertical in the threshold.

**Inverted.** Match the RESOURCE (average set size, smooth and monotone in a scalar shift) and
read the BENEFIT (per-class coverage equity, bounded in [0,1]). Oracle headroom becomes
**+0.31** on worst-class coverage.

**Objective = worst-class coverage, NOT macro.** Macro has no headroom even for an oracle
(measured +0.012): a uniform threshold is already near-optimal for an unweighted mean. That is
Jensen, not a property of delta_y. Macro stays the right statistic for the tail report below.

**Shrunk, with lambda fit on TRAIN classes only.** The raw delta_hat HURTS at realistic
predictor quality, because a worst-class objective is governed by the LARGEST error in
delta_hat, not its variance - and R2 controls mean squared error, so the requirement tightens
as the number of classes grows. Measured: `R2=0.30 -> best lambda 0.10 -> +0.025`, while raw
`lambda=1 -> -0.40`. lambda is a free parameter, so selecting it on the held-out classes would
manufacture a positive result; it is chosen on 𝒴_train and applied unchanged.

**Read the controls, not just the verdict.** `oracle_ceiling` states the headroom that exists,
and `raw_delta_lambda1` states what the retired application would have given.


In [ ]:
from pcc.eval.predictability import ridge_fit, ridge_predict
from pcc.eval.setsize import setsize_translation_shrunk, setsize_translation_heldout_space
from pcc.eval.decomposition import group_quantile

dvec = deltas['primary'][0]
cols = [names.index(f) for f in FEATURE_SETS['stable']]
usable = np.where(np.isfinite(dvec) & np.isfinite(Phi).all(axis=1))[0]
print(f'classes usable for Sec 6.4: {len(usable)}')
qg_emp = group_quantile(s_true_cal, ALPHA, 'empirical')
rng = np.random.default_rng(SEED)

EQ_STAT = 'worst'      # PRIMARY objective; 'macro' has no oracle headroom
acc = {'obs': [], 'null': [], 'oracle': [], 'raw': [], 'lam': []}
for rep in range(20):
    perm = rng.permutation(usable)
    fit_c, held_c = perm[:len(perm)//2], perm[len(perm)//2:]
    if len(held_c) < 5: continue
    model = ridge_fit(Phi[fit_c][:, cols], dvec[fit_c], 1.0)
    dhat = np.zeros(K)
    dhat[held_c] = ridge_predict(model, Phi[held_c][:, cols])
    dhat[fit_c]  = ridge_predict(model, Phi[fit_c][:, cols])   # needed to pick lambda
    dnull = np.array(dhat); dnull[held_c] = rng.permutation(dhat[held_c])
    for tag, dd in (('obs', dhat), ('null', dnull)):
        try:
            r = setsize_translation_shrunk(S_ev, y_ev, ALPHA, None, None,
                                           fit_c, held_c, dd, stat=EQ_STAT,
                                           q_global=qg_emp)
        except ValueError:
            continue
        acc[tag].append(r['delta'][EQ_STAT])
        if tag == 'obs':
            acc['lam'].append(r['lambda_selected_on_train'])
            acc['oracle'].append(r['controls']['oracle_ceiling'])
            acc['raw'].append(r['controls']['raw_delta_lambda1'])

obs = mean_ci(np.array(acc['obs'], float)); nul = mean_ci(np.array(acc['null'], float))
orc = mean_ci(np.array(acc['oracle'], float)); raw = mean_ci(np.array(acc['raw'], float))
beats = bool(obs['ci_low'] > nul['ci_high'])
sec64 = {'stat': EQ_STAT, 'observed': obs, 'shuffled_null': nul,
         'oracle_ceiling': orc, 'raw_delta_lambda1': raw,
         'lambda_mean': float(np.mean(acc['lam'])) if acc['lam'] else float('nan'),
         'beats_null': beats, 'pass': bool(obs['ci_low'] > 0 and beats)}
print(f"[PRIMARY] d {EQ_STAT}-class coverage at matched set size:")
print(f"   observed  {obs['mean']:+.4f} CI [{obs['ci_low']:+.4f},{obs['ci_high']:+.4f}]")
print(f"   shuffled  {nul['mean']:+.4f}   beats_null={beats}")
print(f"   ORACLE CEILING {orc['mean']:+.4f}  <- the headroom that exists at all")
print(f"   raw lambda=1   {raw['mean']:+.4f}  <- what the retired application gives")
print(f"   lambda chosen on TRAIN classes: mean {sec64['lambda_mean']:.3f}")
print(f"   => {'PASS' if sec64['pass'] else 'NOT POSITIVE'}")
if orc['mean'] <= 0.02:
    print('   WARNING: no oracle headroom -> the metric cannot return a positive;')
    print('            do not read the observed value as evidence either way.')


### 8b. SECONDARY - the retired design 4, for continuity with the previous report

Kept so the two designs can be compared on identical data. **Not a verdict.**


In [ ]:
sec64_retired = None
if RUN_RETIRED_64:
    acc2 = {('class_conditional','obs'): [], ('class_conditional','null'): [],
            ('macro','obs'): [], ('macro','null'): []}
    rng2 = np.random.default_rng(SEED)
    for rep in range(20):
        perm = rng2.permutation(usable)
        fit_c, held_c = perm[:len(perm)//2], perm[len(perm)//2:]
        if len(held_c) < 5: continue
        model = ridge_fit(Phi[fit_c][:, cols], dvec[fit_c], 1.0)
        dh = np.zeros(K); dh[held_c] = ridge_predict(model, Phi[held_c][:, cols])
        dn = np.zeros(K); dn[held_c] = rng2.permutation(dh[held_c])
        for obj in ('class_conditional','macro'):
            for tag, dd in (('obs', dh), ('null', dn)):
                try:
                    r = setsize_translation_heldout_space(S_ev, y_ev, ALPHA, qg_emp, dd,
                                                         held_c, objective=obj)
                    acc2[(obj,tag)].append(r['gap'])
                except ValueError:
                    pass
    sec64_retired = {}
    for obj in ('class_conditional','macro'):
        o = mean_ci(np.array(acc2[(obj,'obs')], float))
        nl = mean_ci(np.array(acc2[(obj,'null')], float))
        sec64_retired[obj] = {'observed': o, 'shuffled_null': nl,
                              'beats_null': bool(o['ci_low'] > nl['ci_high'])}
        print(f"   [retired, not a verdict] {obj}: gap={o['mean']:+.3f} null={nl['mean']:+.3f}")
else:
    print('skipped')


## 9. TAIL VIEW - macro-coverage per prevalence stratum

delta_y is unmeasurable for a 2-sample class, but macro-coverage aggregated over hundreds of tail
classes is estimable. This is the only view that says anything about where the claim is supposed to
pay off.


In [ ]:
from pcc.eval.tail import compare_by_stratum

qg = group_quantile(s_true_cal, ALPHA, 'conformal')   # deployment-valid threshold
# Amendment 8: apply the SHRUNK delta_hat here too. The raw vector was measured to
# HURT worst-class coverage (-0.40 at R2~0.3), so reporting the tail under raw
# delta_hat would describe a correction the protocol does not endorse.
LAM = sec64['lambda_mean'] if np.isfinite(sec64['lambda_mean']) else 0.0
dhat_used = LAM * dhat
print(f'applying shrunk delta_hat: lambda={LAM:.3f} (selected on TRAIN classes)')
tail_res = compare_by_stratum(S_ev, y_ev, K, cal_counts, qg, dhat_used,
                              n_strata=N_STRATA, min_count=1, seed=SEED)
print('stratum'.ljust(22) + 'macro_cov unc -> cor'.rjust(24) + 'size unc -> cor'.rjust(22))
for k in tail_res['uncorrected']:
    if k.startswith('_'): continue
    u, c = tail_res['uncorrected'][k], tail_res['corrected'][k]
    print(k.ljust(22) + f"{u['macro_coverage']:.3f} -> {c['macro_coverage']:.3f}".rjust(24)
          + f"{u['avg_set_size']:.2f} -> {c['avg_set_size']:.2f}".rjust(22))
print('\nunevaluable:', tail_res['uncorrected']['_unevaluable'])


## 10. Sec 9 metric bundle (all metrics together, never size alone)


In [ ]:
from pcc.eval.metrics import summary as sec9_summary
from pcc.eval.setsize import corrected_thresholds
from pcc.eval.conformal import build_sets

heldset = set(int(c) for c in held_c)
group_of_class = {y: ('held_out' if y in heldset else 'seen') for y in range(K)}
sec9 = {}
for arm, thr in (('uncorrected', qg), ('corrected', corrected_thresholds(qg, dhat_used))):
    sets = build_sets(S_ev, thr)
    sec9[arm] = sec9_summary(sets, y_ev, K, ALPHA, group_of_class=group_of_class)
    print(arm + ':')
    for k, v in sec9[arm].items(): print(f'    {k:24s} {v}')


## 11. Verdict + report, with every mandated caveat attached


In [ ]:
import time
from pcc.utils.io import write_report

def clean(o):
    if isinstance(o, dict): return {str(k): clean(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [clean(v) for v in o]
    if isinstance(o, (np.floating, np.integer)): return float(o)
    if isinstance(o, (np.bool_,)): return bool(o)
    if isinstance(o, np.ndarray): return None
    return o

prim = gate_bc['primary|stable']['stratified']
gate_B_primary = prim['n_strata_gate_B_pass'] >= max(1, prim['n_strata_reported'] - 1)
abl = prim['n_strata_full_beats_ablation']
gate_C_primary = all(v >= max(1, prim['n_strata_reported'] - 1) for v in abl.values()) if abl else None

print('GATE A:', 'PASS' if gate_A_pass else 'FAIL')
print('GATE B (stratified, primary):', 'PASS' if gate_B_primary else 'FAIL -> AMBIGUOUS per Sec 3.3')
print('GATE C (stratified, primary):', gate_C_primary, '| per-ablation strata:', abl)
nested_flag = any(v.get('distance_baseline_is_nested_in_full') for v in prim['summary'].values())
n_prereg = sum(1 for v in prim['summary'].values() if v.get('gate_C_pass_prereg'))
if nested_flag:
    print(f'  (the primary gate-C distance baseline is NESTED in the full model, so this is'
          f' harder than pre-registered Sec 6.5C;')
    print(f'   against the INDEPENDENT pre-registered baseline: '
          f'{n_prereg}/{prim["n_strata_reported"]} strata pass)')
s64 = sec64['pass']
print(f"Sec 6.4 (d {sec64['stat']}-class coverage at matched size):",
      'PASS' if s64 else 'NOT POSITIVE',
      f"| observed {sec64['observed']['mean']:+.4f} of an oracle ceiling "
      f"{sec64['oracle_ceiling']['mean']:+.4f} "
      f"({100*sec64['observed']['mean']/max(sec64['oracle_ceiling']['mean'],1e-9):.0f}% captured)")

# Sec 6.4 is the OUTCOME the gates are proxies for: gate B/C ask whether geometry
# predicts delta_y, Sec 6.4 asks whether the resulting correction actually helps. When
# the two disagree, say so explicitly rather than letting a proxy override the outcome.
verdict = ('PASS' if (gate_A_pass and gate_B_primary and gate_C_primary
                      and s64) else 'NOT PASSED')
if s64 and not (gate_B_primary and gate_C_primary):
    print()
    print('NOTE - the proxy and the outcome disagree, and both are reported:')
    print('  Sec 6.4 (the OUTCOME: does the predicted correction help?) PASSES,')
    print('  while the stratified gate B/C (a PROXY: is delta_y predictable?) does not.')
    print(f'  The stratified test has {prim["summary"][list(prim["summary"])[0]]["n_train_classes"]}'
          ' training classes per stratum, so its CIs span >0.5 in R2 and it cannot')
    print('  resolve a moderate effect; per Amendment 5 a gate-B FAIL is AMBIGUOUS.')
    print('  This is NOT scored as an overall PASS. A human decides whether an outcome')
    print('  that passes with an underpowered proxy is sufficient (AGENTS.md Sec 12).')
caveats = [
    'Sec 3.3 NOT satisfied: descriptor stability peaked at 0.815 (<0.90), tail quartile flat at',
    '  ~0.68 and unfixable by quota -> a gate-B FAILURE IS AMBIGUOUS, not evidence of no signal.',
    'Descriptor quality tracks prevalence (head-tail spread +0.238) -> gate C is judged on the',
    '  STRATIFIED form; the pooled form cannot separate a geometry story from a prevalence story.',
    'Scores are OURS, not LTC released (checkpoint gate FAILED under a written exception).',
    f'alpha={ALPHA} only; alpha=0.05/0.01 are infeasible for most classes (187/57 of 1081).',
    'Gate A/B/C are computed on an n_cal-restricted, hence PREVALENCE-SELECTED, class subset.',
    'Amendment 7: gate C ablations are resolved from the COMPLETE descriptor set, so they now',
    '  run for the PRIMARY stability-screened set too. The previous run reported',
    '  gate_C_primary_pass=null only because the ablation columns had been sliced away.',
    'A gate-C FAIL in an UNDERPOWERED cell (n_train < 3x n_features) is not evidence: the',
    '  15-feature model had 19 training classes per stratum, vs +0.453 for the 2-feature set.',
    'The stability screen made cos_knn_5 the distance baseline, but cos_knn_5 is ALSO one of',
    '  the two PRIMARY features -- so gate C as scored is a NESTED test (does logit_margin add',
    '  anything beyond cos_knn_5?), strictly harder than pre-registered Sec 6.5C. Both the',
    '  nested and the independent pre-registered verdicts are reported; do not conflate them.',
    '  15-feature model had 19 training classes per stratum, vs +0.453 for the 2-feature set.',
]
print()
for c in caveats: print('CAVEAT:', c)

report = write_report('pcc/reports', f'04_phase1_gate_{DATASET}',
    hypothesis='delta_y is a reliable class-level signal (A), predictable from class geometry (B) '
               'beyond trivial predictors (C), and the predicted correction buys efficiency on '
               'held-out classes (6.4)',
    pass_criteria='A: split-half reliability CI low >= 0.30. B: held-out R2 CI excludes 0 in '
                  'essentially every prevalence stratum (STRATIFIED is primary per Amendment 5); '
                  'a FAIL is recorded as AMBIGUOUS because Sec 3.3 was not met. C: the full '
                  'descriptor beats log-prevalence-only and distance-only within strata. 6.4: '
                  'held-out-label-space gap > 0 and beating a shuffled null. delta_y at matched '
                  'n_cal (25 primary, 10 sensitivity); both feature sets and both pooled and '
                  'stratified views always reported.',
    config=dict(dataset=DATASET, backbone=BACKBONE, alpha=ALPHA,
                n_cal_primary=N_CAL_PRIMARY, n_cal_sensitivity=N_CAL_SENSITIVITY,
                n_splits_A=N_SPLITS_A, n_splits_BC=N_SPLITS_BC, n_strata=N_STRATA,
                feature_sets={k: list(v) for k, v in FEATURE_SETS.items()},
                stable_threshold=STABLE_THRESHOLD,
                scores_source=prov.get('scores_source'),
                under_gate_exception=bool(prov.get('under_gate_exception')),
                amendments=['#amendment-2','#amendment-4','#amendment-5', '#amendment-7', '#amendment-8']),
    seed=SEED,
    results={'gate_A': {'reliability': r_delta, 'ci_low': rel_ci['ci_low'],
                        'ci_high': rel_ci['ci_high'], 'pass': gate_A_pass,
                        'n_classes_eligible': relA['n_classes_eligible'],
                        'n_splits_with_a_value': relA['n_splits_with_a_value'],
                        'n_classes_contributing_mean': relA['n_classes_contributing_mean'],
                        'undefined_reason': relA.get('undefined_reason')},
             'ceilings': {'r_delta': r_delta, 'r_phi': r_phi, 'joint': r_delta*r_phi},
             'delta_y': {t: {'n_cal': deltas[t][2],
                             'n_classes': int((deltas[t][1] & np.isfinite(deltas[t][0])).sum())}
                         for t in deltas},
             'prevalence_null': clean(nulls),
             'gate_BC': clean(gate_bc), 'sec_6_4': clean(sec64),
             'sec_6_4_retired_design4': clean(sec64_retired),
             'tail_by_stratum': clean(tail_res), 'sec_9_metrics': clean(sec9),
             'gate_B_primary_pass': bool(gate_B_primary),
             'gate_C_primary_pass': gate_C_primary,
             'gate_C_primary_nested_baseline': bool(nested_flag),
             'gate_C_primary_strata_pass_prereg_baseline': int(n_prereg),
             'caveats': caveats},
    conclusion=f'{verdict} - see caveats; a gate-B failure here is AMBIGUOUS (Sec 3.3 unmet)',
    started_at=time.time())
print()
print('report:', report)
print('VERDICT:', verdict)
